<a href="https://colab.research.google.com/github/PiotrZielinski3/MEDICA/blob/main/medica_app.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
import sqlite3
import threading
from flask import Flask, request, render_template, redirect, url_for, session
from jinja2 import DictLoader
from google.colab import output

# Inicjalizacja aplikacji Flask
app = Flask(__name__)
app.secret_key = "super_tajny_klucz_medica_plus"
PORT = 5000

# 1. Konfiguracja wbudowanej bazy danych (SQLite)
def init_db():
    conn = sqlite3.connect('medica_plus.db')
    c = conn.cursor()
    c.execute('''
        CREATE TABLE IF NOT EXISTS pacjenci (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            pesel TEXT UNIQUE NOT NULL,
            haslo TEXT NOT NULL,
            imie TEXT NOT NULL,
            nazwisko TEXT NOT NULL,
            historia_chorob TEXT
        )
    ''')
    conn.commit()
    conn.close()

init_db()

# --- SZABLONY HTML ---
HTML_BASE = """
<!DOCTYPE html>
<html lang="pl">
<head>
    <meta charset="UTF-8">
    <title>Sieć Klinik Medica+</title>
    <style>
        body { font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif; background-color: #eef2f5; color: #333; margin: 0; padding: 20px; }
        .container { max-width: 500px; margin: 40px auto; background: white; padding: 30px; border-radius: 10px; box-shadow: 0 4px 8px rgba(0,0,0,0.1); }
        h1, h2 { color: #2c3e50; text-align: center; }
        input[type="text"], input[type="password"] { width: 95%; padding: 10px; margin: 10px 0 20px 0; border: 1px solid #ccc; border-radius: 5px; }
        button { width: 100%; padding: 10px; background-color: #27ae60; color: white; border: none; border-radius: 5px; font-size: 16px; cursor: pointer; }
        button:hover { background-color: #219150; }
        .link { display: block; text-align: center; margin-top: 15px; color: #3498db; text-decoration: none; }
        .link:hover { text-decoration: underline; }
        .error { color: #e74c3c; text-align: center; font-weight: bold; }
        .success { color: #27ae60; text-align: center; font-weight: bold; }
        .card { background: #ecf0f1; padding: 15px; border-left: 5px solid #3498db; margin-bottom: 20px; }
    </style>
</head>
<body>
    <div class="container">
        {% block content %}{% endblock %}
    </div>
</body>
</html>
"""

HTML_LOGIN = """
{% extends 'base' %}
{% block content %}
    <h1>Medica+</h1>
    <h2>Logowanie Pacjenta</h2>
    {% if error %}<p class="error">{{ error }}</p>{% endif %}
    {% if msg %}<p class="success">{{ msg }}</p>{% endif %}
    <form method="POST" action="/login">
        <label>Numer PESEL:</label>
        <input type="text" name="pesel" required>
        <label>Hasło:</label>
        <input type="password" name="haslo" required>
        <button type="submit">Zaloguj się</button>
    </form>
    <a class="link" href="/register">Nowy pacjent? Zarejestruj się</a>
{% endblock %}
"""

HTML_REGISTER = """
{% extends 'base' %}
{% block content %}
    <h1>Medica+</h1>
    <h2>Rejestracja Nowego Pacjenta</h2>
    {% if error %}<p class="error">{{ error }}</p>{% endif %}
    <form method="POST" action="/register">
        <label>Imię:</label>
        <input type="text" name="imie" required>
        <label>Nazwisko:</label>
        <input type="text" name="nazwisko" required>
        <label>Numer PESEL:</label>
        <input type="text" name="pesel" required>
        <label>Hasło:</label>
        <input type="password" name="haslo" required>
        <button type="submit">Dodaj pacjenta</button>
    </form>
    <a class="link" href="/login">Powrót do logowania</a>
{% endblock %}
"""

HTML_DASHBOARD = """
{% extends 'base' %}
{% block content %}
    <h1>Karta Pacjenta</h1>
    <div class="card">
        <h3>Dane osobowe</h3>
        <p><strong>Imię i nazwisko:</strong> {{ pacjent[3] }} {{ pacjent[4] }}</p>
        <p><strong>PESEL:</strong> {{ pacjent[1] }}</p>
    </div>
    <div class="card" style="border-left-color: #e67e22;">
        <h3>Historia chorób i notatki</h3>
        <p>{{ pacjent[5] if pacjent[5] else 'Brak wpisów w historii medycznej.' }}</p>
    </div>
    <a class="link" href="/logout" style="color: #e74c3c;">Wyloguj się</a>
{% endblock %}
"""

app.jinja_loader = DictLoader({
    'base': HTML_BASE,
    'login': HTML_LOGIN,
    'register': HTML_REGISTER,
    'dashboard': HTML_DASHBOARD
})

# --- ROUTING APLIKACJI ---
@app.route('/')
def home():
    if 'pacjent_id' in session:
        return redirect(url_for('dashboard'))
    return redirect(url_for('login'))

@app.route('/register', methods=['GET', 'POST'])
def register():
    if request.method == 'POST':
        imie = request.form['imie']
        nazwisko = request.form['nazwisko']
        pesel = request.form['pesel']
        haslo = request.form['haslo']
        historia = "Zarejestrowano w systemie Medica+."

        conn = sqlite3.connect('medica_plus.db')
        c = conn.cursor()
        try:
            c.execute("INSERT INTO pacjenci (pesel, haslo, imie, nazwisko, historia_chorob) VALUES (?, ?, ?, ?, ?)",
                      (pesel, haslo, imie, nazwisko, historia))
            conn.commit()
            conn.close()
            return render_template('login', msg="Rejestracja udana! Możesz się zalogować.")
        except sqlite3.IntegrityError:
            conn.close()
            return render_template('register', error="Pacjent o podanym numerze PESEL już istnieje w systemie!")

    return render_template('register')

@app.route('/login', methods=['GET', 'POST'])
def login():
    if request.method == 'POST':
        pesel = request.form['pesel']
        haslo = request.form['haslo']

        conn = sqlite3.connect('medica_plus.db')
        c = conn.cursor()
        c.execute("SELECT * FROM pacjenci WHERE pesel = ? AND haslo = ?", (pesel, haslo))
        pacjent = c.fetchone()
        conn.close()

        if pacjent:
            session['pacjent_id'] = pacjent[0]
            return redirect(url_for('dashboard'))
        else:
            return render_template('login', error="Błędny PESEL lub hasło.")

    return render_template('login')

@app.route('/dashboard')
def dashboard():
    if 'pacjent_id' not in session:
        return redirect(url_for('login'))

    conn = sqlite3.connect('medica_plus.db')
    c = conn.cursor()
    c.execute("SELECT * FROM pacjenci WHERE id = ?", (session['pacjent_id'],))
    pacjent = c.fetchone()
    conn.close()

    if not pacjent:
        session.pop('pacjent_id', None)
        return redirect(url_for('login'))

    return render_template('dashboard', pacjent=pacjent)

@app.route('/logout')
def logout():
    session.pop('pacjent_id', None)
    return redirect(url_for('login'))

# --- URUCHOMIENIE SERWERA W COLAB ---
thread = threading.Thread(target=lambda: app.run(host='0.0.0.0', port=PORT, debug=True, use_reloader=False))
thread.daemon = True
thread.start()

# Otwarcie w nowej karcie zamiast iframe
output.serve_kernel_port_as_window(PORT)

Try `serve_kernel_port_as_iframe` instead. 
 * Serving Flask app '__main__'


<IPython.core.display.Javascript object>

 * Debug mode: on


Address already in use
Port 5000 is in use by another program. Either identify and stop that program, or start the server with a different port.
